In [4]:
# =================================================================================================
# Multimodal ResNet-50 with Metadata and 7-Fold CV ≈ 70% Train / 15% Val / 15% Test Overall
# - Concise subgroup reports (Gender, Age buckets, Skin tone)
# - Fairness metrics: WGA, EOD with min-support filtering
# - Pooled (weighted) subgroup overall accuracies across folds
# =================================================================================================

import os, time, json, warnings
from copy import deepcopy
from collections import defaultdict
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, accuracy_score, cohen_kappa_score,
    matthews_corrcoef, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from torch.amp import GradScaler, autocast

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

# Optional FLOPs (best effort)
try:
    from thop import profile
    HAVE_THOP = True
except Exception:
    HAVE_THOP = False


# ---------------------------------------------
# 1) CONFIG — UPDATE THESE TWO PATHS
# ---------------------------------------------
class Config:
    DATA_DIR = r"D:\AIUB\Neural\Synthetic_Skin_Disease_Dataset"      # <-- update to your root image dir
    ORIGINAL_METADATA_CSV = r"D:\AIUB\Neural\complete_metadata_m.csv"  # <-- update to your CSV path

    RESULTS_DIR = "results_Multimodal_ResNet50_KFold"
    MODEL_NAME = "ResNet-50" # Changed from DINOv3
    IMG_SIZE = 224
    BATCH_SIZE = 128
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 2e-5
    EPOCHS = 200
    NUM_CLASSES = 5
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP = True
    NUM_WORKERS = 0
    PATIENCE = 20
    WARMUP_EPOCHS = 10
    LABEL_SMOOTHING = 0.1
    PARTIALLY_UNFREEZE = False # set True to unfreeze last block of ResNet-50 (layer4)

    # K-Fold settings for ~70/15/15 overall
    N_SPLITS = 7                # ~1/7 ≈ 14.3% Test per fold (close to 15%)
    TARGET_TEST_FRAC = 0.15     # target overall (for docs)
    TARGET_VAL_FRAC  = 0.15     # target overall (for docs)

    # Inside each fold's remainder (~85%), choose Val so overall ≈ 15%
    # val_within_remainder = TARGET_VAL_FRAC / (1 - 1/N_SPLITS)
    VAL_WITHIN_REMAINDER = 0.15 / (1.0 - 1.0/7.0) # ≈ 0.175

    # Metadata settings
    META_DROPOUT = 0.3
    META_HIDDEN_DIM = 128
    META_OUT_DIM = 256

    # Fairness: minimum subgroup size for stable WGA/EOD & pooled tables
    MIN_SUPPORT = 15

config = Config()
os.makedirs(config.RESULTS_DIR, exist_ok=True)
print("Device:", config.DEVICE)
print(f"Model: {config.MODEL_NAME}")
print(f"K-Fold: N_SPLITS={config.N_SPLITS} (~Test {100/config.N_SPLITS:.1f}%), Val_in_remainder={config.VAL_WITHIN_REMAINDER:.3f}")


# ---------------------------------------------
# 2) CSV STANDARDIZATION
# ---------------------------------------------
def standardize_metadata_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Accepts a CSV with columns like:
      filepath, split, label, age, dominant_gender, dominant_race
    and returns standardized columns:
      filename, label, age_group, gender, skin_tone
    Keeps/deduces split if provided; K-Fold will override per fold.
    """
    df = df.copy()

    if 'filepath' in df.columns:
        df['filepath'] = df['filepath'].astype(str).str.replace('\\', '/', regex=False)

    # filename
    if 'filename' not in df.columns:
        if 'filepath' in df.columns:
            df['filename'] = df['filepath'].apply(lambda p: os.path.basename(str(p)))
        else:
            raise ValueError("CSV must include 'filepath' or 'filename'.")

    # label
    if 'label' not in df.columns and 'filepath' in df.columns:
        df['label'] = df['filepath'].str.split('/').str[1]
    df['label'] = df['label'].astype(str)

    # age_group (bucket by numeric age if present)
    if 'age_group' not in df.columns:
        if 'age' in df.columns:
            age_num = pd.to_numeric(df['age'], errors='coerce')
            bins    = [-1, 17, 29, 44, 64, 200]  # child, young_adult, adult, middle_aged, senior
            names   = ['child', 'young_adult', 'adult', 'middle_aged', 'senior']
            df['age_group'] = pd.cut(age_num, bins=bins, labels=names).astype(str).fillna('unknown')
        else:
            df['age_group'] = 'unknown'

    # gender
    if 'gender' not in df.columns:
        if 'dominant_gender' in df.columns:
            g = df['dominant_gender'].astype(str).str.strip().str.lower()
            df['gender'] = g.map({'man':'male','male':'male','woman':'female','female':'female'}).fillna('unknown')
        else:
            df['gender'] = 'unknown'

    # skin_tone / race
    if 'skin_tone' not in df.columns:
        if 'dominant_race' in df.columns:
            df['skin_tone'] = df['dominant_race'].astype(str).str.strip().str.lower()
        else:
            df['skin_tone'] = 'unknown'

    required = ['filename', 'label', 'age_group', 'gender', 'skin_tone']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after standardization: {missing}")

    return df


# ---------------------------------------------
# 3) DATASET & VOCAB
# ---------------------------------------------
class GPUAugOnlyNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                              std=[0.229, 0.224, 0.225])
    @torch.no_grad()
    def forward(self, x): return self.normalize(x)

class MetaVocab:
    def __init__(self, df_train, cols=('age_group','gender','skin_tone')):
        self.cols = cols
        self.unk_token = "<UNK>"
        self.vocabs = {}
        for c in cols:
            uniq = sorted([self.unk_token] + sorted(df_train[c].dropna().astype(str).unique().tolist()))
            self.vocabs[c] = {v:i for i,v in enumerate(uniq)}
        self.sizes = {c: len(self.vocabs[c]) for c in cols}
    def encode_row(self, row):
        out = {}
        for c in self.cols:
            v = str(row[c]) if pd.notnull(row[c]) else self.unk_token
            out[c] = self.vocabs[c].get(v, self.vocabs[c][self.unk_token])
        return out

class MultimodalDermDataset(Dataset):
    """
    Uses 'filepath' if present (relative to DATA_DIR or absolute),
    else DATA_DIR/split/label/filename (split provided per fold).
    """
    def __init__(self, config, df, class_to_idx, meta_vocab, transform=None):
        self.config = config
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.meta_vocab = meta_vocab

        self.image_paths, self.labels, self.meta = [], [], []
        use_filepath = 'filepath' in self.df.columns
        for _, row in self.df.iterrows():
            if use_filepath and pd.notnull(row.get('filepath', None)):
                raw = str(row['filepath'])
                p = raw if os.path.isabs(raw) else os.path.join(config.DATA_DIR, raw)
            else:
                # Expect a 'split' column already assigned for this fold
                p = os.path.join(config.DATA_DIR, str(row['split']), str(row['label']), str(row['filename']))
            self.image_paths.append(p)
            self.labels.append(self.class_to_idx[str(row['label'])])
            self.meta.append(self.meta_vocab.encode_row(row))

        # Preload images (disable if memory constrained)
        self.images = []
        for p in self.image_paths:
            with Image.open(p).convert("RGB") as im:
                self.images.append(im.copy())

        self.class_names = sorted(class_to_idx.keys())

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        image = self.images[idx]
        if self.transform: image = self.transform(image)
        return image, self.labels[idx], self.meta[idx]


# ---------------------------------------------
# 4) MODEL (ResNet-50 + Metadata Fusion) -- MODIFIED SECTION
# ---------------------------------------------
class MetadataEncoder(nn.Module):
    def __init__(self, meta_vocab, emb_dims=None, hidden=128, out_dim=256, dropout=0.1):
        super().__init__()
        sizes = meta_vocab.sizes
        if emb_dims is None:
            emb_dims = {
                'age_group':  min(16, max(4, sizes['age_group'])),
                'gender':     min(8,  max(3, sizes['gender'])),
                'skin_tone':  min(16, max(4, sizes['skin_tone'])),
            }
        self.emb_age = nn.Embedding(sizes['age_group'], emb_dims['age_group'])
        self.emb_gen = nn.Embedding(sizes['gender'],    emb_dims['gender'])
        self.emb_tone= nn.Embedding(sizes['skin_tone'], emb_dims['skin_tone'])
        concat_dim = emb_dims['age_group'] + emb_dims['gender'] + emb_dims['skin_tone']
        self.mlp = nn.Sequential(
            nn.Linear(concat_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim), nn.GELU()
        )
    def forward(self, meta_idx):
        a = self.emb_age(meta_idx['age_group'])
        g = self.emb_gen(meta_idx['gender'])
        s = self.emb_tone(meta_idx['skin_tone'])
        return self.mlp(torch.cat([a,g,s], dim=-1))

class ResNet50Multimodal(nn.Module):
    """
    A multimodal model using a pretrained ResNet-50 backbone.
    Features from the image backbone are concatenated with features
    from the metadata encoder and passed to a final classifier.
    """
    def __init__(self, num_classes, meta_vocab, meta_hidden_dim, meta_out_dim, meta_dropout):
        super().__init__()
        # Image backbone (ResNet-50)
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        
        # Freeze all backbone parameters initially
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # Unfreeze the final residual block (layer4) if configured
        if config.PARTIALLY_UNFREEZE:
            for param in self.backbone.layer4.parameters():
                param.requires_grad = True
        
        num_ftrs = self.backbone.fc.in_features  # 2048 for ResNet-50
        self.backbone.fc = nn.Identity()  # Remove original classifier

        # Metadata encoder
        self.meta_encoder = MetadataEncoder(meta_vocab, 
                                            hidden=meta_hidden_dim, 
                                            out_dim=meta_out_dim,
                                            dropout=meta_dropout)

        # Fusion and Classifier Head
        self.classifier = nn.Sequential(
            nn.Linear(num_ftrs + meta_out_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, meta_idx):
        # Image stream
        self.backbone.eval() if not config.PARTIALLY_UNFREEZE else self.backbone.train()
        if config.PARTIALLY_UNFREEZE:
            img_features = self.backbone(images)
        else:
            with torch.no_grad():
                img_features = self.backbone(images)

        # Metadata stream
        meta_features = self.meta_encoder(meta_idx)

        # Fusion by concatenation
        combined_features = torch.cat((img_features, meta_features), dim=1)

        # Classification
        output = self.classifier(combined_features)
        return output


# ---------------------------------------------
# 5) TRAIN / EVAL HELPERS
# ---------------------------------------------
def _to_device_meta(batch_meta, device):
    if isinstance(batch_meta, dict):
        ag = torch.as_tensor(batch_meta['age_group'], dtype=torch.long, device=device)
        ge = torch.as_tensor(batch_meta['gender'],    dtype=torch.long, device=device)
        st = torch.as_tensor(batch_meta['skin_tone'], dtype=torch.long, device=device)
        return {'age_group': ag, 'gender': ge, 'skin_tone': st}
    else:
        ag = torch.tensor([m['age_group'] for m in batch_meta], dtype=torch.long, device=device)
        ge = torch.tensor([m['gender']    for m in batch_meta], dtype=torch.long, device=device)
        st = torch.tensor([m['skin_tone'] for m in batch_meta], dtype=torch.long, device=device)
        return {'age_group': ag, 'gender': ge, 'skin_tone': st}

def make_dataloaders_from_df(config, df_std):
    classes = sorted(df_std[df_std['split']=='Train']['label'].astype(str).unique())
    if not classes:
        classes = sorted(df_std['label'].astype(str).unique())
    class_to_idx = {c:i for i,c in enumerate(classes)}
    config.NUM_CLASSES = len(classes)

    df_vocab = df_std[df_std['split']=='Train'] if (df_std['split']=='Train').any() else df_std
    meta_vocab = MetaVocab(df_vocab, cols=('age_group','gender','skin_tone'))

    train_tfm = transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.2,0.1)], p=0.8),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
    ])
    eval_tfm = transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
    ])

    dsets = {}
    for split, tfm in [('Train', train_tfm), ('Val', eval_tfm), ('Test', eval_tfm)]:
        dfi = df_std[df_std['split'] == split]
        dsets[split] = MultimodalDermDataset(config, dfi, class_to_idx, meta_vocab, transform=tfm)

    dls = {
        'Train': DataLoader(dsets['Train'], batch_size=config.BATCH_SIZE, shuffle=True,
                            num_workers=config.NUM_WORKERS, pin_memory=True),
        'Val':   DataLoader(dsets['Val'],   batch_size=config.BATCH_SIZE, shuffle=False,
                            num_workers=config.NUM_WORKERS, pin_memory=True),
        'Test':  DataLoader(dsets['Test'],  batch_size=config.BATCH_SIZE, shuffle=False,
                            num_workers=config.NUM_WORKERS, pin_memory=True),
    }
    sizes = {k: len(dsets[k]) for k in dsets}
    return dls, sizes, classes, meta_vocab

def train_model(model, dls, sizes, num_epochs):
    start = time.time()
    best = deepcopy(model.state_dict())
    best_val = float('inf')
    patience = 0
    hist = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

    scaler = GradScaler(enabled=config.USE_AMP)
    gpu_norm = GPUAugOnlyNorm().to(config.DEVICE)
    opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    def lr_lambda(ep):
        if ep < config.WARMUP_EPOCHS: return float(ep+1)/float(config.WARMUP_EPOCHS)
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * (ep - config.WARMUP_EPOCHS) / (num_epochs - config.WARMUP_EPOCHS))))
    sch = optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)

    print("\n" + "="*80 + f"\nSTART TRAINING (Multimodal {config.MODEL_NAME}) for {num_epochs} epochs\n" + "="*80)
    for ep in range(num_epochs):
        for phase in ['Train','Val']:
            model.train() if phase=='Train' else model.eval()
            run_loss, run_correct = 0.0, 0
            for images, labels, metas in dls[phase]:
                images = images.to(config.DEVICE)
                labels = torch.as_tensor(labels, dtype=torch.long, device=config.DEVICE)
                meta_idx = _to_device_meta(metas, config.DEVICE)
                images = gpu_norm(images)
                opt.zero_grad(set_to_none=True)
                with autocast(device_type=config.DEVICE.split(':')[0], dtype=torch.float16, enabled=config.USE_AMP):
                    logits = model(images=images, meta_idx=meta_idx)
                    loss = criterion(logits, labels)
                if phase=='Train':
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
                preds = logits.argmax(1)
                run_loss += loss.item() * images.size(0)
                run_correct += (preds == labels).sum().item()
            e_loss = run_loss / max(1, sizes[phase])
            e_acc  = run_correct / max(1, sizes[phase])
            if phase=='Train':
                hist['train_loss'].append(e_loss); hist['train_acc'].append(e_acc)
            else:
                hist['val_loss'].append(e_loss);   hist['val_acc'].append(e_acc)
                if e_loss < best_val:
                    best_val = e_loss; best = deepcopy(model.state_dict()); patience = 0
                    torch.save(model.state_dict(), os.path.join(config.RESULTS_DIR, 'best_multimodal_resnet.pth'))
                else:
                    patience += 1
        sch.step()
        print(f"Epoch {ep+1}/{num_epochs} | Train {hist['train_loss'][-1]:.4f}/{hist['train_acc'][-1]:.4f} | "
              f"Val {hist['val_loss'][-1]:.4f}/{hist['val_acc'][-1]:.4f} | LR {sch.get_last_lr()[0]:.2e}")
        if patience >= config.PATIENCE:
            print("Early stopping.")
            break
    model.load_state_dict(best)
    print("Training complete in {:.1f} minutes. Best Val Loss {:.4f}".format((time.time()-start)/60.0, best_val))
    return model, hist


# ---------------------------------------------
# 6) EVALUATION (Concise reports + WGA/EOD with min-support + Pooled subgroup accuracies)
# ---------------------------------------------
def _print_section(title):
    print("\n" + "="*80)
    print(title.upper())
    print("="*80 + "\n")

def _print_group_report(group_title, y_true, y_pred, class_names):
    labels_full = list(range(len(class_names)))
    print(f"--- Results for Group: {group_title} ---")
    print(classification_report(y_true, y_pred,
                                labels=labels_full,
                                target_names=class_names,
                                zero_division=0))
    print()

def _group_fairness_metrics(y_true, y_pred, groups, num_classes, min_support=None):
    uniq_groups = sorted(pd.Series(groups).astype(str).unique())
    per_group_acc, per_group_tprs, counts = {}, {}, {}
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)

    for g in uniq_groups:
        mask = (np.asarray(groups) == g)
        counts[g] = int(mask.sum())
        yt = y_true[mask]; yp = y_pred[mask]
        acc_g = (yt == yp).mean() if len(yt) > 0 else np.nan
        tprs = []
        for c in range(num_classes):
            tp = ((yt == c) & (yp == c)).sum()
            pos = (yt == c).sum()
            tprs.append(tp/pos if pos > 0 else np.nan)
        per_group_acc[g] = acc_g
        per_group_tprs[g] = np.array(tprs, dtype=float)

    if min_support is not None:
        filt_keys = [g for g in uniq_groups if counts[g] >= min_support]
    else:
        filt_keys = uniq_groups

    if filt_keys:
        wga = float(np.nanmin([per_group_acc[g] for g in filt_keys]))
    else:
        wga = float('nan')

    def macro_eod(keys, per_group_tprs, K):
        max_gap, worst = 0.0, (None, None)
        for i in range(len(keys)):
            for j in range(i+1, len(keys)):
                gi, gj = keys[i], keys[j]
                a, b = per_group_tprs[gi], per_group_tprs[gj]
                mask = ~(np.isnan(a) | np.isnan(b))
                if mask.sum() == 0: 
                    continue
                gap = float(np.nanmean(np.abs(a[mask] - b[mask])))
                if gap > max_gap:
                    max_gap, worst = gap, (gi, gj)
        return max_gap, worst

    eod_val, worst_pair = macro_eod(filt_keys, per_group_tprs, num_classes)

    return {
        'per_group_acc': per_group_acc,
        'per_group_tprs': per_group_tprs,
        'counts': counts,
        'keys_used': filt_keys,
        'wga': wga,
        'eod': eod_val,
        'worst_pair': worst_pair
    }

def evaluate_and_visualize(model, dl_test, class_names, df_fold):
    model.eval()
    gpu_norm = GPUAugOnlyNorm().to(config.DEVICE)
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for images, labels, metas in dl_test:
            images = images.to(config.DEVICE)
            labels = torch.as_tensor(labels, dtype=torch.long, device=config.DEVICE)
            meta_idx = _to_device_meta(metas, config.DEVICE)
            images = gpu_norm(images)
            logits = model(images=images, meta_idx=meta_idx)
            probs  = torch.softmax(logits, dim=1)
            y_true += labels.cpu().numpy().tolist()
            y_pred += logits.argmax(1).cpu().numpy().tolist()
            y_prob += probs.cpu().numpy().tolist()

    y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)

    # OVERALL (concise)
    _print_section("OVERALL PERFORMANCE (ON COMBINED DATASET)")
    print(classification_report(y_true, y_pred,
                                labels=list(range(len(class_names))),
                                target_names=class_names,
                                zero_division=0))

    # Align Test metadata rows
    df_test = df_fold[df_fold['split'] == 'Test'][['age_group','gender','skin_tone']].reset_index(drop=True)
    if len(df_test) != len(y_true):
        m = min(len(df_test), len(y_true))
        df_test = df_test.iloc[:m].copy()
        y_true = y_true[:m]; y_pred = y_pred[:m]; y_prob = y_prob[:m]

    # BY GENDER
    _print_section("PERFORMANCE ANALYSIS BY: DOMINANT_GENDER")
    for g_key, g_title in [('male','Man'), ('female','Woman')]:
        mask = (df_test['gender'].astype(str) == g_key).values
        if mask.sum() == 0: continue
        _print_group_report(g_title, y_true[mask], y_pred[mask], class_names)

    # BY AGE BUCKETS
    _print_section("PERFORMANCE ANALYSIS BY: AGE_GROUP")
    def bucket(x):
        x = str(x)
        if x == 'child': return '0-17'
        if x in ('young_adult','adult','middle_aged'): return '18-59'
        if x == 'senior': return '60+'
        return 'unknown'
    age_disp = df_test['age_group'].apply(bucket)
    for group in ['0-17','18-59','60+']:
        mask = (age_disp == group).values
        if mask.sum()==0: continue
        _print_group_report(group, y_true[mask], y_pred[mask], class_names)

    # BY SKIN TONE
    _print_section("PERFORMANCE ANALYSIS BY: DOMINANT_RACE")
    for tone in sorted(df_test['skin_tone'].astype(str).unique()):
        if tone == 'unknown': continue
        mask = (df_test['skin_tone'].astype(str) == tone).values
        if mask.sum()==0: continue
        _print_group_report(tone, y_true[mask], y_pred[mask], class_names)

    # ---------- FAIRNESS SUMMARY ----------
    groups_composite = (age_disp.astype(str) + "|" +
                        df_test['gender'].astype(str) + "|" +
                        df_test['skin_tone'].astype(str)).values

    fair_all = _group_fairness_metrics(
        y_true, y_pred, groups_composite, num_classes=len(class_names), min_support=None
    )
    fair_filt = _group_fairness_metrics(
        y_true, y_pred, groups_composite, num_classes=len(class_names), min_support=config.MIN_SUPPORT
    )

    print("\n" + "="*80)
    print(f"FAIRNESS SUMMARY (ALL GROUPS)  |  WGA: {fair_all['wga']:.4f}   EOD (max TPR gap): {fair_all['eod']:.4f}")
    if fair_all['worst_pair'][0] is not None:
        print(f"Worst pair: {fair_all['worst_pair'][0]}  vs  {fair_all['worst_pair'][1]}")
    print("="*80)

    print("\n" + "="*80)
    print(f"FAIRNESS SUMMARY (FILTERED, n >= {config.MIN_SUPPORT})  |  "
          f"WGA: {fair_filt['wga']:.4f}   EOD (max TPR gap): {fair_filt['eod']:.4f}")
    if fair_filt['worst_pair'][0] is not None:
        print(f"Worst pair (among filtered): {fair_filt['worst_pair'][0]}  vs  {fair_filt['worst_pair'][1]}")
    print("Groups used (filtered):", fair_filt['keys_used'])
    print("="*80)

    # Scalar metrics
    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    try:
        y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))
        roc_auc_macro = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
    except Exception:
        roc_auc_macro = float('nan')

    # ---------- NEW: per-subgroup accuracy & counts for pooling across folds ----------
    def acc_counts_from_series(series_vals):
        """returns dict: group -> (correct, total)"""
        d = {}
        s = series_vals.astype(str).values
        for g in np.unique(s):
            mask = (s == g)
            tot = int(mask.sum())
            if tot == 0: continue
            cor = int((y_true[mask] == y_pred[mask]).sum())
            d[str(g)] = (cor, tot)
        return d

    pooled_info = {
        'gender':    acc_counts_from_series(df_test['gender']),
        'age':       acc_counts_from_series(age_disp),
        'skin_tone': acc_counts_from_series(df_test['skin_tone']),
        'composite': acc_counts_from_series(pd.Series(groups_composite)),
    }

    return {
        'accuracy': acc, 'roc_auc_macro': roc_auc_macro, 'kappa': kappa, 'mcc': mcc,
        'wga_all': fair_all['wga'], 'eod_all': fair_all['eod'],
        'wga_filtered': fair_filt['wga'], 'eod_filtered': fair_filt['eod'],
        'pooled_groups': pooled_info  # NEW: for cross-fold pooled subgroup accuracy
    }


# ---------------------------------------------
# 7) K-FOLD: BUILD SPLITS -> TRAIN -> EVAL -> AGGREGATE
# ---------------------------------------------
def build_kfold_splits(df_std, n_splits=7, seed=42, val_within_remainder=0.1765):
    """
    For each fold:
      - Test = 1/n_splits of data (stratified by label)  ~15%
      - From the remainder (~85%), Val = val_within_remainder of remainder  (so overall ~15%)
      - Train = the rest (~70%)
    Returns a list of fold DataFrames with 'split' = Train/Val/Test.
    """
    df = df_std.copy().reset_index(drop=True)
    y = df['label'].astype(str).values
    idx = np.arange(len(df))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = []
    for fold_id, (trainval_idx, test_idx) in enumerate(skf.split(idx, y), 1):
        rest_labels = y[trainval_idx]
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_within_remainder, random_state=seed+fold_id)
        for tr_rel, val_rel in sss2.split(trainval_idx, rest_labels):
            tr_idx = trainval_idx[tr_rel]
            val_idx = trainval_idx[val_rel]

        split = pd.Series(index=np.arange(len(df)), data="Ignore", dtype=str)
        split.loc[tr_idx]  = "Train"
        split.loc[val_idx] = "Val"
        split.loc[test_idx]= "Test"

        dfi = df.copy()
        dfi['split'] = split.values
        dfi = dfi[dfi['split'].isin(['Train','Val','Test'])].reset_index(drop=True)

        # Quick fold report
        nT = (dfi.split=='Train').sum(); nV = (dfi.split=='Val').sum(); nE = (dfi.split=='Test').sum()
        print(f"Fold {fold_id}: Train={nT} ({100*nT/len(dfi):.1f}%)  Val={nV} ({100*nV/len(dfi):.1f}%)  Test={nE} ({100*nE/len(dfi):.1f}%)")
        folds.append(dfi)
    return folds

def run_one_fold(fold_df, fold_id, config):
    print("\n" + "#"*88)
    print(f"### FOLD {fold_id} ###  Train={sum(fold_df.split=='Train')}, Val={sum(fold_df.split=='Val')}, Test={sum(fold_df.split=='Test')}")
    print("#"*88)

    dls, sizes, class_names, meta_vocab = make_dataloaders_from_df(config, fold_df)
    
    # --- MODEL INSTANTIATION CHANGED ---
    model = ResNet50Multimodal(num_classes=len(class_names),
                               meta_vocab=meta_vocab,
                               meta_hidden_dim=config.META_HIDDEN_DIM,
                               meta_out_dim=config.META_OUT_DIM,
                               meta_dropout=config.META_DROPOUT).to(config.DEVICE)

    # Optional model size / FLOPs
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params: {total_params/1e6:.2f}M | Trainable: {trainable_params/1e6:.2f}M ({100*trainable_params/total_params:.2f}%)")
    if HAVE_THOP:
        try:
            dummy_img = torch.randn(1,3,config.IMG_SIZE,config.IMG_SIZE).to(config.DEVICE)
            dummy_meta = {
                'age_group': torch.zeros(1, dtype=torch.long, device=config.DEVICE),
                'gender':    torch.zeros(1, dtype=torch.long, device=config.DEVICE),
                'skin_tone': torch.zeros(1, dtype=torch.long, device=config.DEVICE),
            }
            macs, params = profile(model, inputs=(dummy_img, dummy_meta), verbose=False)
            print(f"Approx MACs: {macs/1e9:.2f} G, Params (THOP): {params/1e6:.2f} M")
        except Exception as e:
            print("THOP profile failed (skipped):", e)

    model, _ = train_model(model, dls, sizes, num_epochs=config.EPOCHS)

    best_path = os.path.join(config.RESULTS_DIR, 'best_multimodal_resnet.pth')
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=config.DEVICE))

    metrics = evaluate_and_visualize(model, dls['Test'], class_names, fold_df)
    return metrics

def kfold_cross_val(df_std, config):
    folds = build_kfold_splits(df_std,
                               n_splits=config.N_SPLITS,
                               seed=42,
                               val_within_remainder=config.VAL_WITHIN_REMAINDER)
    per_fold = []
    for i, fold_df in enumerate(folds, 1):
        m = run_one_fold(fold_df, i, config)
        m['fold'] = i
        per_fold.append(m)

    perf = pd.DataFrame(per_fold).set_index('fold')
    def mean_std(col):
        vals = perf[col].astype(float).values
        return float(np.nanmean(vals)), float(np.nanstd(vals))

    summary = {
        'acc_mean': mean_std('accuracy')[0], 'acc_std': mean_std('accuracy')[1],
        'roc_macro_mean': mean_std('roc_auc_macro')[0], 'roc_macro_std': mean_std('roc_auc_macro')[1],
        'kappa_mean': mean_std('kappa')[0], 'kappa_std': mean_std('kappa')[1],
        'mcc_mean': mean_std('mcc')[0], 'mcc_std': mean_std('mcc')[1],
        'wga_all_mean': mean_std('wga_all')[0], 'wga_all_std': mean_std('wga_all')[1],
        'eod_all_mean': mean_std('eod_all')[0], 'eod_all_std': mean_std('eod_all')[1],
        'wga_filt_mean': mean_std('wga_filtered')[0], 'wga_filt_std': mean_std('wga_filtered')[1],
        'eod_filt_mean': mean_std('eod_filtered')[0], 'eod_filt_std': mean_std('eod_filtered')[1],
    }

    _print_section("OVERALL K-FOLD RESULTS")
    print(f"Accuracy         : {summary['acc_mean']:.4f} ± {summary['acc_std']:.4f}")
    print(f"ROC AUC (Macro)  : {summary['roc_macro_mean']:.4f} ± {summary['roc_macro_std']:.4f}")
    print(f"Cohen's Kappa    : {summary['kappa_mean']:.4f} ± {summary['kappa_std']:.4f}")
    print(f"MCC              : {summary['mcc_mean']:.4f} ± {summary['mcc_std']:.4f}")
    print("-"*40)
    print(f"WGA (All Groups) : {summary['wga_all_mean']:.4f} ± {summary['wga_all_std']:.4f}")
    print(f"EOD (All Groups) : {summary['eod_all_mean']:.4f} ± {summary['eod_all_std']:.4f}")
    print(f"WGA (n>={config.MIN_SUPPORT}) : {summary['wga_filt_mean']:.4f} ± {summary['wga_filt_std']:.4f}")
    print(f"EOD (n>={config.MIN_SUPPORT}) : {summary['eod_filt_mean']:.4f} ± {summary['eod_filt_std']:.4f}")
    print()

    # ---- POOLED SUBGROUP ACCURACY TABLE ----
    pooled_counts = defaultdict(lambda: {'correct': 0, 'total': 0})
    for _, row in perf.iterrows():
        for group_type, group_data in row['pooled_groups'].items():
            for group_name, (correct, total) in group_data.items():
                key = f"{group_type}:{group_name}"
                pooled_counts[key]['correct'] += correct
                pooled_counts[key]['total'] += total
    
    pooled_results = []
    for key, data in pooled_counts.items():
        if data['total'] >= config.MIN_SUPPORT:
            acc = data['correct'] / data['total'] if data['total'] > 0 else 0
            pooled_results.append({
                'Group': key,
                'Accuracy': acc,
                'Support': data['total']
            })
            
    df_pooled = pd.DataFrame(pooled_results).sort_values(by=['Group']).reset_index(drop=True)
    if not df_pooled.empty:
        _print_section(f"POOLED SUBGROUP ACCURACY ACROSS ALL FOLDS (SUPPORT >= {config.MIN_SUPPORT})")
        print(df_pooled.to_string())

    # Save summary and per-fold results
    with open(os.path.join(config.RESULTS_DIR, "summary_results.json"), "w") as f:
        json.dump(summary, f, indent=4)
    perf.drop(columns=['pooled_groups']).to_csv(os.path.join(config.RESULTS_DIR, "per_fold_results.csv"))
    if not df_pooled.empty:
        df_pooled.to_csv(os.path.join(config.RESULTS_DIR, "pooled_subgroup_accuracy.csv"), index=False)
    
    print(f"\nResults saved to '{config.RESULTS_DIR}'")
    return summary, perf, df_pooled


# ---------------------------------------------
# 8) MAIN EXECUTION
# ---------------------------------------------
if __name__ == '__main__':
    try:
        df_orig = pd.read_csv(config.ORIGINAL_METADATA_CSV)
        df_std = standardize_metadata_columns(df_orig)
        print(f"Loaded and standardized metadata: {len(df_std)} rows.")
        
        summary, per_fold_results, pooled_df = kfold_cross_val(df_std, config)

    except FileNotFoundError:
        print(f"ERROR: Metadata file not found at '{config.ORIGINAL_METADATA_CSV}'")
        print("Please update the 'ORIGINAL_METADATA_CSV' path in the Config class.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()

Device: cuda
Model: ResNet-50
K-Fold: N_SPLITS=7 (~Test 14.3%), Val_in_remainder=0.175
Loaded and standardized metadata: 565 rows.
Fold 1: Train=399 (70.6%)  Val=85 (15.0%)  Test=81 (14.3%)
Fold 2: Train=399 (70.6%)  Val=85 (15.0%)  Test=81 (14.3%)
Fold 3: Train=399 (70.6%)  Val=85 (15.0%)  Test=81 (14.3%)
Fold 4: Train=399 (70.6%)  Val=85 (15.0%)  Test=81 (14.3%)
Fold 5: Train=399 (70.6%)  Val=85 (15.0%)  Test=81 (14.3%)
Fold 6: Train=400 (70.8%)  Val=85 (15.0%)  Test=80 (14.2%)
Fold 7: Train=400 (70.8%)  Val=85 (15.0%)  Test=80 (14.2%)

########################################################################################
### FOLD 1 ###  Train=399, Val=85, Test=81
########################################################################################
Params: 24.73M | Trainable: 1.22M (4.93%)
Approx MACs: 4.13 G, Params (THOP): 24.73 M

START TRAINING (Multimodal ResNet-50) for 200 epochs
Epoch 1/200 | Train 1.6921/0.2206 | Val 1.6023/0.2706 | LR 2.00e-05
Epoch 2/200 | Train 1.6355